In [1]:
import boto3
import os

# 1. Conexión directa al puerto de la API de MinIO
s3_client = boto3.client(
    's3',
    endpoint_url='http://minio:9000',
    aws_access_key_id='minioadmin',
    aws_secret_access_key='minioadminpassword',
    region_name='us-east-1'
)

buckets_requeridos = ["bronze", "silver", "gold"]
namespace = "smart_grids"
local_dir = "/app/data_lakehouse"

print("--- INICIANDO CREACIÓN DE BUCKETS DESDE PYTHON ---")

# 2. Crear Buckets de forma real y verificable
for bucket in buckets_requeridos:
    try:
        # Forzar validación de existencia
        s3_client.head_bucket(Bucket=bucket)
        print(f" El bucket '{bucket}' ya existe.")
    except Exception:
        print(f" Creando bucket '{bucket}' desde Python...")
        s3_client.create_bucket(Bucket=bucket)
        
    # Crear el prefijo/namespace estructurado inyectando el placeholder
    s3_client.put_object(Bucket=bucket, Key=f"{namespace}/.keep", Body=b"init")

print("\n--- INICIANDO CARGA AUTOMÁTICA DE ARCHIVOS CSV (BRONZE) ---")

# 3. Escanear y subir los archivos locales de tu laptop
if os.path.exists(local_dir):
    files_to_upload = [f for f in os.listdir(local_dir) if f.endswith('.csv')]
    
    if not files_to_upload:
        print(f" Advertencia: No hay archivos .csv en la ruta montada: {local_dir}")
        
    for file_name in files_to_upload:
        local_path = os.path.join(local_dir, file_name)
        s3_key = f"{namespace}/{file_name}"
        
        print(f" Subiendo {file_name} -> s3://bronze/{s3_key}...")
        s3_client.upload_file(local_path, "bronze", s3_key)
        
    print("\n¡Pipeline de inicialización completado con éxito!")
else:
    print(f" Error crítico: La ruta {local_dir} no está accesible en el contenedor. Revisa los volúmenes.")


--- INICIANDO CREACIÓN DE BUCKETS DESDE PYTHON ---
 Creando bucket 'bronze' desde Python...
 Creando bucket 'silver' desde Python...
 Creando bucket 'gold' desde Python...

--- INICIANDO CARGA AUTOMÁTICA DE ARCHIVOS CSV (BRONZE) ---
 Subiendo 08_distribution_networks.csv -> s3://bronze/smart_grids/08_distribution_networks.csv...
 Subiendo 04_energy_storage.csv -> s3://bronze/smart_grids/04_energy_storage.csv...
 Subiendo 10_data_mgmt_systems.csv -> s3://bronze/smart_grids/10_data_mgmt_systems.csv...
 Subiendo 13_smart_meters.csv -> s3://bronze/smart_grids/13_smart_meters.csv...
 Subiendo 11_ami_head_ends.csv -> s3://bronze/smart_grids/11_ami_head_ends.csv...
 Subiendo 09_distribution_transformers.csv -> s3://bronze/smart_grids/09_distribution_transformers.csv...
 Subiendo 07_power_transformers.csv -> s3://bronze/smart_grids/07_power_transformers.csv...
 Subiendo 05_scada_dms.csv -> s3://bronze/smart_grids/05_scada_dms.csv...
 Subiendo 12_consumers.csv -> s3://bronze/smart_grids/12_cons